# LSTM (sequence model)\n\nGoal: predict **next weekend gap %** using a **sequence of previous weekends** for each ticker.\n\nThis notebook builds a per-ticker time series of weekend samples, then trains an LSTM on sequences of length `SEQ_LEN`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent.parent  # notebooks/models -> repo root
WEEKEND_CSV = REPO_ROOT / "structured_csv_data_files" / "Weekend_dataset.csv"
WEEKEND_CSV

In [ ]:
w = pd.read_csv(WEEKEND_CSV)
w["Date_parsed"] = pd.to_datetime(w["Date"], errors="coerce", utc=True)
w["weekday_num"] = w["Date_parsed"].dt.weekday

for c in ["Open", "Close", "High", "Low", "Volume"]:
    if c in w.columns:
        w[c] = pd.to_numeric(w[c], errors="coerce")

w = w.dropna(subset=["Ticker", "Date_parsed"]).sort_values(["Ticker", "Date_parsed"]).reset_index(drop=True)

w["next_weekday"] = w.groupby("Ticker")["weekday_num"].shift(-1)
w["next_date"] = w.groupby("Ticker")["Date_parsed"].shift(-1)
w["next_open"] = w.groupby("Ticker")["Open"].shift(-1)

pairs = w.loc[(w["weekday_num"] == 4) & (w["next_weekday"] == 0)].copy()
delta_days = (pairs["next_date"] - pairs["Date_parsed"]).dt.days
pairs = pairs.loc[(delta_days >= 3) & (delta_days <= 5)].copy()

pairs["y_gap_pct"] = (pairs["next_open"] - pairs["Close"]) / pairs["Close"]
pairs["monday_date"] = pairs["next_date"]

pairs[["Ticker", "Date_parsed", "monday_date", "y_gap_pct"]].head()

In [ ]:
# Build feature matrix from Friday-only numeric columns
exclude = {
    "Date",
    "Date_parsed",
    "weekday_num",
    "next_weekday",
    "next_date",
    "next_open",
    "monday_date",
    "y_gap_pct",
}

X_num = pairs[[c for c in pairs.columns if c not in exclude]].select_dtypes(include=["number"]).copy()
y = pd.to_numeric(pairs["y_gap_pct"], errors="coerce")

X_num.shape, y.describe()

## Sequence building\n\nWe create sequences per ticker, ordered by Friday date.\nEach training example uses the previous `SEQ_LEN` rows of features to predict the next row's `y_gap_pct`.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 8  # tune (e.g., 4, 8, 12)

# Impute + scale (fit only on train later; here we just prep placeholders)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# We'll do a time-based split by monday_date first
cutoff = pairs["monday_date"].quantile(0.8)
train_mask = pairs["monday_date"] <= cutoff

X_train_raw = X_num.loc[train_mask]
X_test_raw = X_num.loc[~train_mask]

X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

X_train_scaled.shape, X_test_scaled.shape, cutoff

In [ ]:
# Reattach scaled arrays back to the original pairs rows (by index)
pairs_train = pairs.loc[train_mask].copy()
pairs_test = pairs.loc[~train_mask].copy()

pairs_train["_row"] = np.arange(len(pairs_train))
pairs_test["_row"] = np.arange(len(pairs_test))

feat_names = X_num.columns.to_list()

def build_sequences(pairs_df: pd.DataFrame, X_scaled: np.ndarray, seq_len: int):
    X_seq = []
    y_seq = []
    meta = []

    for ticker, g in pairs_df.sort_values(["Ticker", "Date_parsed"]).groupby("Ticker"):
        idx = g.index.to_numpy()
        rows = g["_row"].to_numpy()
        if len(rows) <= seq_len:
            continue

        for j in range(seq_len, len(rows)):
            hist_rows = rows[j - seq_len : j]
            X_seq.append(X_scaled[hist_rows])
            y_seq.append(float(g.iloc[j]["y_gap_pct"]))
            meta.append((ticker, g.iloc[j]["Date_parsed"], g.iloc[j]["monday_date"]))

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32), meta

X_tr, y_tr, meta_tr = build_sequences(pairs_train, X_train_scaled, SEQ_LEN)
X_te, y_te, meta_te = build_sequences(pairs_test, X_test_scaled, SEQ_LEN)

X_tr.shape, X_te.shape

## LSTM model\n\nRequires TensorFlow/Keras (`pip install tensorflow`).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential(
    [
        layers.Input(shape=(SEQ_LEN, X_tr.shape[-1])),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ]
)

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=[keras.metrics.MeanAbsoluteError()])
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
]

history = model.fit(
    X_tr,
    y_tr,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
pred = model.predict(X_te).ravel()

mae = np.mean(np.abs(pred - y_te))
rmse = float(np.sqrt(np.mean((pred - y_te) ** 2)))

print("MAE:", mae)
print("RMSE:", rmse)

# Quick scatter of predictions
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
plt.scatter(y_te, pred, alpha=0.3)
plt.xlabel("true y_gap_pct")
plt.ylabel("pred y_gap_pct")
plt.title("LSTM predictions")
plt.axline((0, 0), slope=1, color="black", linewidth=1)
plt.tight_layout()
plt.show()